# 07 — Inventory Optimization

Safety stock is derived from the spread of **lead-time forecast error**, not from historical demand variability. That is the entire reason forecasting has value: a better forecast shrinks the error spread and therefore the stock required. Sizing from demand variability returns the same answer no matter how good the forecast is.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
warnings.filterwarnings('ignore')

from src.config import load_config
from src.data.loader import load_panel
cfg = load_config()
panel = load_panel(cfg)
print(f'panel: {len(panel):,} rows, {panel.series_id.nunique()} series, {panel.date.nunique()} days')

In [ ]:
from src.inventory.policy import PolicyParams, z_for_service_level
p = PolicyParams.from_config(cfg)
print(f'lead time {p.lead_time_days} d + review {p.review_period_days} d = {p.protection_days} d of exposure')
print('z(95%) =', round(z_for_service_level(0.95), 4))

Under periodic review the exposure is lead time **plus** the review period: a stockout can occur any time before the *next* order arrives. Omitting the review period systematically under-protects.

## Lead-time forecast error and safety stock

In [ ]:
import pandas as pd
from src.inventory.policy import lead_time_forecast_errors, error_spread, build_policy
proc = cfg.resolve('data','processed_dir')
ew = pd.read_parquet(proc / 'error_windows.parquet')
lt = lead_time_forecast_errors(ew, 'lightgbm', p.protection_days)
sp = error_spread(lt)
print('windows per series (median):', int(sp.n_windows.median()))
sp[['error_std','error_q90','error_q95']].describe().round(3)

In [ ]:
test_preds = pd.read_parquet(proc / 'test_predictions.parquet')
pol = build_policy(cfg, test_preds, 'lightgbm', service_level=0.95, lt_errors=lt)
pol[['series_id','expected_lt_demand','error_std','safety_stock_normal',
     'safety_stock_empirical','reorder_point_normal']].head(10).round(2)

## Simulate: does the policy deliver its service level?

A reorder point from a formula is a claim. Replaying the held-out demand day by day is the test. Unmet demand is treated as **lost sales**, not backorders.

In [ ]:
from src.inventory.simulate import simulate_policy, compare_policies
from src.inventory.policy import baseline_policy
from src.data.loader import test_fold
tf = test_fold(cfg)
base = baseline_policy(cfg, panel[panel.date <= tf.origin], 0.95)
sims = [simulate_policy(panel, base, tf.start, tf.end, p.lead_time_days,
                        p.review_period_days, 'reorder_point_normal',
                        policy_name='baseline_demand_variability'),
        simulate_policy(panel, pol, tf.start, tf.end, p.lead_time_days,
                        p.review_period_days, 'reorder_point_normal',
                        policy_name='forecast_normal_ss')]
compare_policies(sims).round(4)

## Cost the policies

> Only unit value comes from the data. Margin, holding rate, ordering cost and the stockout penalty are declared assumptions.

In [ ]:
from src.inventory import costs as C
uv = C.unit_values(panel, as_of=tf.origin)
econ = C.Economics.from_config(cfg)
pd.DataFrame([{'policy': s.policy_name,
               **C.cost_summary(C.cost_simulation(s.per_series, uv, econ, days=28))}
              for s in sims]).round(2)